In [1]:
import numpy as np
import tensorflow as tf
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report, confusion_matrix

In [2]:
# ------------------------------------------------------------
# Step 1: Load and preprocess the MNIST dataset
# ------------------------------------------------------------
def load_and_preprocess():
    """
    Loads MNIST from TensorFlow, normalizes pixel values,
    reshapes images for CNN input, one-hot encodes labels,
    and explicitly creates training, validation, and test sets.
    """
    # Load dataset: 60,000 training images, 10,000 test images
    (X_full, y_full), (X_test, y_test) = tf.keras.datasets.mnist.load_data()
    # Normalize pixel values from [0, 255] â†’ [0, 1]
    X_full = X_full / 255.0
    X_test = X_test / 255.0
    # Reshape to add channel dimension (required for CNNs)
    # Shape: (samples, height, width, channels)
    X_full = X_full.reshape(-1, 28, 28, 1)
    X_test = X_test.reshape(-1, 28, 28, 1)
    # One-hot encode labels for multi-class classification (0â€“9)
    y_full = tf.keras.utils.to_categorical(y_full, num_classes=10)
    y_test = tf.keras.utils.to_categorical(y_test, num_classes=10)
    # Explicit split of original training data:
    # 90% â†’ training, 10% â†’ validation
    X_train, X_val, y_train, y_val = train_test_split(
    X_full, y_full, test_size=0.1, random_state=42
    )
    return X_train, X_val, X_test, y_train, y_val, y_test

In [3]:
# ------------------------------------------------------------
# Step 2: Build the Convolutional Neural Network (CNN)
# ------------------------------------------------------------
def build_cnn(input_shape):
    """
    Builds and compiles a CNN model with convolution,
    pooling, and fully connected layers.
    """
    model = tf.keras.Sequential([
        # First convolutional block
        tf.keras.layers.Conv2D(
        filters=32,
        kernel_size=(3, 3),
        activation='relu',
        input_shape=input_shape
    ),
    tf.keras.layers.MaxPooling2D(pool_size=(2, 2)),

        # Second convolutional block
        tf.keras.layers.Conv2D(
        filters=64,
        kernel_size=(3, 3),
        activation='relu'
    ),
    tf.keras.layers.MaxPooling2D(pool_size=(2, 2)),

    # Flatten feature maps into a vector
    tf.keras.layers.Flatten(),
    # Fully connected layer
    tf.keras.layers.Dense(128, activation='relu'),
    # Output layer: 10 neurons for digits 0â€“9
    tf.keras.layers.Dense(10, activation='softmax')
    ])

    # Compile the model
    model.compile(
        optimizer='adam',
        loss='categorical_crossentropy',
        metrics=['accuracy']
    )
    return model

In [4]:
# ------------------------------------------------------------
# Step 3: Train the CNN model
# ------------------------------------------------------------
def train_model(model, X_train, y_train, X_val, y_val,
                epochs=5, batch_size=32):
 """
 Trains the CNN model and monitors validation performance.
 """
 model.fit(
    X_train,
    y_train,
    validation_data=(X_val, y_val),
    epochs=epochs,
    batch_size=batch_size
 )


In [5]:
# ------------------------------------------------------------
# Step 4: Evaluate the trained model
# ------------------------------------------------------------
def evaluate_model(model, X_test, y_test):
    """
    Evaluates the model on the test dataset and prints
    classification metrics and confusion matrix.
    """
    # Predict class probabilities
    y_pred = model.predict(X_test)
    # Convert probabilities to class labels
    y_pred_classes = np.argmax(y_pred, axis=1)
    y_true = np.argmax(y_test, axis=1)
    # Display classification metrics
    print("\nClassification Report:\n")
    print(classification_report(y_true, y_pred_classes))
    # Display confusion matrix
    print("\nConfusion Matrix:\n")
    print(confusion_matrix(y_true, y_pred_classes))


In [6]:
import warnings
warnings.filterwarnings("ignore")
# ------------------------------------------------------------
# Main function: orchestrates the full pipeline
# ------------------------------------------------------------
def main():
    """
    Main execution function:
    - Loads and preprocesses data
    - Builds CNN
    - Trains model
    - Evaluates performance
    """
    # Load and preprocess MNIST data
    X_train, X_val, X_test, y_train, y_val, y_test = load_and_preprocess()

    # Build CNN model
    model = build_cnn(input_shape=(28, 28, 1))

    # Train CNN model
    train_model(model, X_train, y_train, X_val, y_val)

    # Evaluate model on test data
    evaluate_model(model, X_test, y_test)


# ------------------------------------------------------------
# Entry point
# ------------------------------------------------------------
if __name__ == "__main__":
 main()

Epoch 1/5
1688/1688 ━━━━━━━━━━━━━━━━━━━━ 52s 30ms/step - accuracy: 0.9022 - loss: 0.3166 - val_accuracy: 0.9835 - val_loss: 0.0562
Epoch 2/5
1688/1688 ━━━━━━━━━━━━━━━━━━━━ 51s 30ms/step - accuracy: 0.9857 - loss: 0.0459 - val_accuracy: 0.9893 - val_loss: 0.0386
Epoch 3/5
1688/1688 ━━━━━━━━━━━━━━━━━━━━ 50s 30ms/step - accuracy: 0.9906 - loss: 0.0282 - val_accuracy: 0.9907 - val_loss: 0.0341
Epoch 4/5
1688/1688 ━━━━━━━━━━━━━━━━━━━━ 82s 30ms/step - accuracy: 0.9939 - loss: 0.0199 - val_accuracy: 0.9900 - val_loss: 0.0340
Epoch 5/5
1688/1688 ━━━━━━━━━━━━━━━━━━━━ 81s 30ms/step - accuracy: 0.9954 - loss: 0.0143 - val_accuracy: 0.9898 - val_loss: 0.0401
313/313 ━━━━━━━━━━━━━━━━━━━━ 3s 8ms/step

Classification Report:

              precision    recall  f1-score   support

           0       0.99      0.99      0.99       980
           1       1.00      1.00      1.00      1135
           2       0.99      0.99      0.99      1032
           3       0.99      0.99      0.99      1010
        